In [ ]:
"""
Schema: https://github.com/alibaba/clusterdata/blob/master/cluster-trace-v2018/schema.txt
### batch task
+----------------------------------------------------------------------------------------+
| task_name       | string     |       | task name. unique within a job                  |
| instance_num    | bigint     |       | number of instances                             |
| job_name        | string     |       | job name                                        |
| task_type       | string     |       | task type                                       |
| status          | string     |       | task status                                     |
| start_time      | bigint     |       | start time of the task                          |
| end_time        | bigint     |       | end of time the task                            |
| plan_cpu        | double     |       | number of cpu needed by the task, 100 is 1 core |
| plan_mem        | double     |       | normalized memorty size, [0, 100]               |
+----------------------------------------------------------------------------------------+
Download data: http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/batch_task.tar.gz

Load workflows between Jobid 100 and 150 (Exclude non-workflow jobs with only one task).
Save to workflows_ali2018_origin.json
"""

In [ ]:
import pandas as pd
import tarfile
import io

def read_csv_from_tar_gz_with_pandas(path: str,column_names) -> pd.DataFrame:
    with tarfile.open(path, 'r:gz') as tar:
        member = tar.getmembers()[0]
        
        with tar.extractfile(member) as f:
            csv_string = f.read().decode('utf-8')
            csv_file_like_object = io.StringIO(csv_string)
            df = pd.read_csv(csv_file_like_object,header=None, names=column_names)
            
            return df
df = read_csv_from_tar_gz_with_pandas('batch_task.tar.gz',[
    'task_name',
    'instance_num',
    'job_name',
    'task_type',
    'status',
    'start_time',
    'end_time',
    'plan_cpu',
    'plan_mem',
])
print(df.head())
df['job_id']=df['job_name'].str.split('_').str[1].astype(int)

  task_name  instance_num job_name  task_type      status  start_time  \
0        M1           1.0      j_1          1  Terminated      419912   
1      R2_1           1.0      j_2          1  Terminated       87076   
2        M1           1.0      j_2          1  Terminated       87076   
3      R6_3         371.0      j_3          1  Terminated      157297   
4    J4_2_3        1111.0      j_3          1  Terminated      157329   

   end_time  plan_cpu  plan_mem  
0    419912     100.0      0.20  
1     87086      50.0      0.20  
2     87083      50.0      0.20  
3    157325     100.0      0.49  
4    157376     100.0      0.59  


In [12]:
df[(df['job_id'] > 100) & (df['job_id'] < 150)]['job_name'].unique()

array(['j_102', 'j_106', 'j_110', 'j_113', 'j_119', 'j_121', 'j_126',
       'j_127', 'j_129', 'j_130', 'j_131', 'j_137', 'j_138', 'j_139',
       'j_142', 'j_144', 'j_145', 'j_146', 'j_147', 'j_148', 'j_103',
       'j_104', 'j_105', 'j_107', 'j_109', 'j_111', 'j_112', 'j_116',
       'j_117', 'j_120', 'j_122', 'j_125', 'j_132', 'j_136', 'j_101',
       'j_108', 'j_114', 'j_115', 'j_118', 'j_123', 'j_124', 'j_128',
       'j_133', 'j_134', 'j_135', 'j_140', 'j_141', 'j_143', 'j_149'],
      dtype=object)

In [2]:
def topsort(G):
    count = {u:sum([u in v for v in G.values()]) for u in G}
    Q = [u for u in G if count[u] == 0]
    S = []
    while Q:
        u = Q.pop()
        S.insert(0,u)
        for v in G[u]:
            count[v]-=1
            if count[v]==0:
                Q.append(v)
    return S
workflows_v1 = []
for key,job in df[(df['job_id'] > 100) & (df['job_id'] < 150)].groupby('job_name'):
    if len(job) > 1:
        print(f"Job Name: {key}, Number of Tasks: {len(job)}")
        tasks = {}
        edges = []
        for task in job.itertuples():
            arr = task.task_name[1:].split('_')
            if arr[0]=='ask':
                task_id = arr[1]
                preds = []
            else:
                task_id,preds = arr[0],arr[1:]
            tasks[task_id] = {
                'workload':task.end_time - task.start_time,
                'cpu':task.plan_cpu,
                'mem':task.plan_mem,
            }
            for pred in preds:
                edges.append((pred,task_id))
        all_tasks = set(tasks.keys())
        leaf_tasks = all_tasks - set(e[0] for e in edges)
        orphan_tasks = all_tasks - set(e[1] for e in edges)
        tasks['ENTRY'] = {'workload':0,'cpu':0,'mem':0}
        tasks['EXIT'] = {'workload':0,'cpu':0,'mem':0}
        for t in leaf_tasks:
            edges.append((t,'EXIT'))
        for t in orphan_tasks:
            edges.append(('ENTRY',t))
        G={i:[] for i in tasks.keys()}
        for d in edges:
            G[d[1]].append(d[0]) # G[dst].append(G[src])
        sorted_task_names = topsort(G)
        workflows_v1.append({
            'name':key,
            'source':f"alibaba_cluster-trace-v2018",
            'tasks':[
                {'task_id':task_id,**tasks[name]}
                for task_id,name in enumerate(sorted_task_names)
            ],
            'dataflows':[
                {
                    'src':sorted_task_names.index(src_name),
                    'dst':sorted_task_names.index(dst_name),
                    'datasize':(tasks[src_name]['workload']+tasks[dst_name]['workload'])//2,
                }
                for src_name,dst_name in edges
            ],
        })
        #print(job)
import json
with open('workflows_ali2018_origin.json','w') as f:
    json.dump(workflows_v1,f,ensure_ascii=False,indent=2)

Job Name: j_102, Number of Tasks: 16
Job Name: j_106, Number of Tasks: 3
Job Name: j_107, Number of Tasks: 3
Job Name: j_108, Number of Tasks: 7
Job Name: j_119, Number of Tasks: 2
Job Name: j_123, Number of Tasks: 6
Job Name: j_125, Number of Tasks: 3
Job Name: j_127, Number of Tasks: 16
Job Name: j_128, Number of Tasks: 3
Job Name: j_129, Number of Tasks: 3
Job Name: j_130, Number of Tasks: 4
Job Name: j_131, Number of Tasks: 3
Job Name: j_133, Number of Tasks: 4
Job Name: j_135, Number of Tasks: 3
Job Name: j_136, Number of Tasks: 3
Job Name: j_137, Number of Tasks: 2
Job Name: j_139, Number of Tasks: 12
Job Name: j_140, Number of Tasks: 3
Job Name: j_141, Number of Tasks: 2
Job Name: j_142, Number of Tasks: 3
Job Name: j_144, Number of Tasks: 3
Job Name: j_145, Number of Tasks: 4
Job Name: j_146, Number of Tasks: 8
Job Name: j_149, Number of Tasks: 2
